##  Notebook 4 — Evaluation Metrics & Visualization

---

### Overview
This notebook evaluates a trained Pneumonia detection model on the test set and visualizes key performance metrics.

### Objectives
- Compute classification metrics (Accuracy, Precision, Recall, F1, AUC).  
- Generate and save confusion matrix and ROC curve visualizations.  
- Export a classification report for further analysis.

### Contents
- Load test dataset and trained model  
- Run inference to obtain y_true, y_pred, and y_probs  
- Compute and print metrics  
- Plot & save confusion matrix and ROC curve  
- Save classification report to outputs/results/metrics.json

---


# --- 1. Imports ---

In [5]:


import os, sys
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from collections import Counter

import sys, os


PROJECT_PATH = os.path.abspath("..")   # go one folder up
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)




# --- 3. Import custom modules ---
from src.dataset import ChestXRayDataset
from src.model import PneumoniaCNN
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_roc_curve, save_classification_report
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")

 Using device: cpu


In [ ]:

# --- 5. Load the test dataset ---
test_dataset = ChestXRayDataset(data_dir=f"{PROJECT_PATH}/data", split="test")
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# --- 6. Load the trained model ---
model = PneumoniaCNN()
model_path = os.path.join(PROJECT_PATH, "outputs/models/best_model.pth")
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
print(f" Loaded model from: {model_path}")

# --- 7. Evaluate the model ---
y_true, y_pred, y_probs = evaluate_model(model, test_loader, device)

# --- 8. Compute metrics ---
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)
auc  = roc_auc_score(y_true, y_probs)

print("\n Performance Metrics:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC:       {auc:.4f}")

# --- 9. Plot confusion matrix ---
results_dir = os.path.join(PROJECT_PATH, "outputs/results")
plot_confusion_matrix(y_true, y_pred, results_dir)

# --- 10. Plot ROC curve ---
plot_roc_curve(y_true, y_probs, results_dir)

# --- 11. Save classification report ---
report = save_classification_report(y_true, y_pred, results_dir)
print("\n Classification report saved to outputs/results/metrics.json")

# --- 12. Display the report ---
import pandas as pd
df = pd.DataFrame(report).T
print(df)
